## Objective

Ingest the six raw CSV files and transform them into persistent Delta tables in Databricks.

In [0]:
# Verify the uploaded files
raw_path = "/Volumes/workspace/default/instacart_raw2"

display(dbutils.fs.ls(raw_path))


path,name,size,modificationTime
dbfs:/Volumes/workspace/default/instacart_raw2/aisles.csv,aisles.csv,2603,1784891077000
dbfs:/Volumes/workspace/default/instacart_raw2/departments.csv,departments.csv,270,1784891078000
dbfs:/Volumes/workspace/default/instacart_raw2/order_products__prior.csv,order_products__prior.csv,577550706,1784891206000
dbfs:/Volumes/workspace/default/instacart_raw2/order_products__train.csv,order_products__train.csv,24680147,1784891117000
dbfs:/Volumes/workspace/default/instacart_raw2/orders.csv,orders.csv,108968645,1784891205000
dbfs:/Volumes/workspace/default/instacart_raw2/products.csv,products.csv,2166953,1784891081000


In [0]:
# Test reading departments.csv
departments_df = (
    spark.read
    .option("header", True)     # Use the first row as column names
    .option("inferSchema", True)# Automatically infer column data types
    .csv(f"{raw_path}/departments.csv")
)

display(departments_df)

department_id,department
1,frozen
2,other
3,bakery
4,produce
5,alcohol
6,international
7,beverages
8,pets
9,dry goods pasta
10,bulk


In [0]:
# Store imported tables under workspace.imported_data
from pyspark.sql import functions as F

raw_path = "/Volumes/workspace/default/instacart_raw2"

catalog_name = "workspace"
imported_data_schema_name = "imported_data"
# Create the imported_data schema
spark.sql(
    f"CREATE SCHEMA IF NOT EXISTS {catalog_name}.{imported_data_schema_name}"
)

print("imported_data schema created successfully.")

imported_data schema created successfully.


In [0]:
# Define an explicit schema for each CSV file
# This ensures consistent data types across ingestion runs.
sources = {
    "aisles": {
        "filename": "aisles.csv",
        "schema": "aisle_id INT, aisle STRING"
    },

    "departments": {
        "filename": "departments.csv",
        "schema": "department_id INT, department STRING"
    },

    "products": {
        "filename": "products.csv",
        "schema": """
            product_id INT,
            product_name STRING,
            aisle_id INT,
            department_id INT
        """
    },

    "orders": {
        "filename": "orders.csv",
        "schema": """
            order_id INT,
            user_id INT,
            eval_set STRING,
            order_number INT,
            order_dow INT,
            order_hour_of_day INT,
            days_since_prior_order DOUBLE
        """
    },

    "order_products_prior": {
        "filename": "order_products__prior.csv",
        "schema": """
            order_id INT,
            product_id INT,
            add_to_cart_order INT,
            reordered INT
        """
    },

    "order_products_train": {
        "filename": "order_products__train.csv",
        "schema": """
            order_id INT,
            product_id INT,
            add_to_cart_order INT,
            reordered INT
        """
    }
}

In [0]:
# Read the six CSV files.
# _source_file identifies the source CSV file.
# _ingested_at records when the data was ingested.
imported_data_dfs = {}

for table_name, configuration in sources.items():

    filename = configuration["filename"]
    schema = configuration["schema"]
    file_path = f"{raw_path}/{filename}"

    df = (
        spark.read
        .format("csv")
        .option("header", "true")
        .option("enforceSchema", "false")
        .schema(schema)
        .load(file_path)
        .withColumn("_source_file", F.lit(filename))
        .withColumn("_ingested_at", F.current_timestamp
        # Record the timestamp at which Databricks ingested the data
                    ())
    )

    imported_data_dfs[table_name] = df

    print(f"{table_name} loaded successfully")

aisles loaded successfully
departments loaded successfully
products loaded successfully
orders loaded successfully
order_products_prior loaded successfully
order_products_train loaded successfully


In [0]:
# Until this point, the ingested data exists only as Spark DataFrames.
# Persist the six DataFrames as managed Delta tables for durable storage and downstream processing.
for table_name, df in imported_data_dfs.items():

    full_table_name = (
        f"{catalog_name}.{imported_data_schema_name}.{table_name}"
    )

    (
        df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(full_table_name)
    )

    print(f"Created: {full_table_name}")

Created: workspace.imported_data.aisles
Created: workspace.imported_data.departments
Created: workspace.imported_data.products
Created: workspace.imported_data.orders
Created: workspace.imported_data.order_products_prior
Created: workspace.imported_data.order_products_train


In [0]:
#Verify tables exist
display(
    spark.sql("SHOW TABLES IN workspace.imported_data")
)

database,tableName,isTemporary
imported_data,aisles,false
imported_data,departments,false
imported_data,order_products_prior,false
imported_data,order_products_train,false
imported_data,orders,false
imported_data,products,false


CSV DataFrame in memory
        vs
workspace.imported_data permanent table

In [0]:
from pyspark.sql import functions as F

comparison_results = []

for table_name, source_df in imported_data_dfs.items():

    # Number of rows before persistence:
    # Spark DataFrame created from the source CSV
    rows_before = source_df.count()

    # Fully qualified Bronze Delta table name
    full_table_name = f"workspace.imported_data.{table_name}"

    # Number of rows after persistence:
    # Permanent Delta table
    rows_after = spark.table(full_table_name).count()

    # Calculate the difference between the two row counts
    difference = rows_after - rows_before

    # Validation status
    status = "OK" if difference == 0 else "Review required"

    comparison_results.append(
        (
            table_name,
            rows_before,
            rows_after,
            difference,
            status
        )
    )

# Create a summary DataFrame
row_count_comparison_df = spark.createDataFrame(
    comparison_results,
    [
        "table_name",
        "rows_before",
        "rows_after",
        "difference",
        "status"
    ]
)

display(
    row_count_comparison_df.orderBy(
        F.desc("rows_before")
    )
)

table_name,rows_before,rows_after,difference,status
order_products_prior,32434489,32434489,0,OK
orders,3421083,3421083,0,OK
order_products_train,1384617,1384617,0,OK
products,49688,49688,0,OK
aisles,134,134,0,OK
departments,21,21,0,OK


## Pipeline

Raw Kaggle CSV Files  
↓  
Unity Catalog Volume  
↓  
Explicit Spark Schemas  
↓  
Spark DataFrames (`imported_data`)  
↓  
Managed Delta Tables (`imported_data`)

---

### Bronze Layer

Raw data is preserved as received from the source.

Example:

`departments.csv` → `workspace.imported_data.departments`

---

### What Happens During Bronze Ingestion

The raw CSV files are read using Spark with explicitly defined column names and data types. The resulting DataFrames are persisted as Delta tables.

Technical metadata columns are also added to support traceability:

- `_source_file`: identifies the original source file.
- `_ingested_at`: records when the data was ingested.

At the Bronze layer, the source data is intentionally preserved without business-level transformations. Null values, duplicates, and potential data-quality issues are retained for validation and processing in subsequent layers.

**Example:**

`_source_file = departments.csv`  
`_ingested_at = 2026-07-24 10:30:00`